# Hierarchical semantic clustering of animal sentences

This notebook walks through the complete Leiden AI pipeline on a synthetic, labelled corpus of 96 animal sentences. The reference labels form a three-level taxonomy—**habitat → animal class → species**—but they are never passed to the clustering algorithms. They are used only after clustering to interpret the unsupervised result.

The pipeline runs on CPU and is deliberately decomposed into its five implementation stages:

1. sentence embeddings;
2. HNSW approximate-neighbor indexing;
3. weighted k-nearest-neighbor graph construction;
4. recursive Leiden community detection;
5. centroid-based representative selection.


## 1. Imports and reproducibility

Run this notebook from either the repository root or the `notebooks` directory. The helper below locates the project root and makes its `src` package importable.


In [ ]:
import json
import os
import sys
from collections import Counter
from pathlib import Path

import numpy as np

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Run this notebook from the repository or notebooks directory.")

sys.path.insert(0, str(PROJECT_ROOT))

from src.cluster import add_representatives, node_to_dict, print_tree
from src.hnsw import (
    EmbeddingConfig,
    HNSWConfig,
    LeidenConfig,
    build_hnsw_index,
    embed_sentences,
)
from src.knn_graph import build_knn_graph
from src.leiden import hierarchical_leiden

np.random.seed(42)
print(f"Project root: {PROJECT_ROOT}")
print("Execution device: CPU")


## 2. Load and inspect the synthetic corpus

The JSON file stores one sentence per record together with reference metadata. Repeating key animal and habitat concepts across varied natural-language sentences creates enough local and global structure to illustrate recursive clustering without feeding labels into the model.


In [ ]:
dataset_path = PROJECT_ROOT / "notebooks" / "animal_sentences.json"
records = json.loads(dataset_path.read_text(encoding="utf-8"))
sentences = [record["sentence"] for record in records]

print(f"Records: {len(records)}")
for field in ("habitat", "animal_class", "species"):
    counts = Counter(record[field] for record in records)
    print(f"{field:>12}: {len(counts):>2} groups | {dict(counts)}")

print("\nExample records:")
for record in records[:3]:
    print(f"  {record['habitat']:>11} / {record['animal_class']:<8} / "
          f"{record['species']:<9} — {record['sentence']}")


## 3. Configure the demonstration

The CPU-friendly MiniLM model keeps the example quick. The graph and Leiden settings are tuned for this small, intentionally structured corpus. In particular, the low initial resolution finds broad groups, while the multiplier asks deeper recursive calls to seek finer communities.


In [ ]:
embedding_config = EmbeddingConfig(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    batch_size=32,
    device="cpu",
    normalize=True,
)

hnsw_config = HNSWConfig(
    k=8,
    M=16,
    ef_construction=100,
    ef_search=64,
    min_similarity=0.28,
    num_threads=2,
)

leiden_config = LeidenConfig(
    resolution=0.10,
    resolution_multiplier=3.0,
    max_depth=3,
    min_cluster_size=6,
    min_child_size=2,
    n_iterations=4,
    seed=42,
)

print(embedding_config)
print(hnsw_config)
print(leiden_config)


## 4. Encode sentences as semantic vectors

`SentenceTransformer` produces one 384-dimensional vector per sentence. Normalization makes each vector unit length, allowing cosine similarity to be computed with a dot product.


In [ ]:
embeddings = embed_sentences(sentences, embedding_config)

norms = np.linalg.norm(embeddings, axis=1)
print(f"Shape: {embeddings.shape}")
print(f"Dtype: {embeddings.dtype}")
print(f"Vector norm range: {norms.min():.4f}–{norms.max():.4f}")

# A quick semantic sanity check using exact cosine similarities.
for left, right in [(0, 1), (0, 40), (64, 65)]:
    similarity = float(embeddings[left] @ embeddings[right])
    print(f"cosine({left:>2}, {right:>2}) = {similarity:.3f}")


## 5. Build the HNSW index

The index organizes the vectors as a navigable small-world graph. It supports fast approximate cosine-neighbor queries and prevents graph construction from requiring all pairwise comparisons.


In [ ]:
hnsw_index = build_hnsw_index(embeddings, hnsw_config)

labels, distances = hnsw_index.knn_query(embeddings[:1], k=5)
print("Nearest sentences to record 0:")
for index, distance in zip(labels[0], distances[0]):
    print(f"  similarity={1.0 - float(distance):.3f} | {sentences[int(index)]}")


## 6. Construct the weighted k-NN graph

Each sentence becomes a vertex. Approximate neighbor links above the cosine threshold become weighted, undirected edges. Duplicate directional links are collapsed to a single edge with the strongest observed similarity.


In [ ]:
graph = build_knn_graph(embeddings, hnsw_index, hnsw_config)

weights = np.asarray(graph.es["weight"], dtype=np.float32)
degrees = np.asarray(graph.degree(), dtype=np.int64)
print(f"Vertices: {graph.vcount()}")
print(f"Edges: {graph.ecount()}")
print(f"Density: {graph.density():.4f}")
print(f"Degree range: {degrees.min()}–{degrees.max()}")
print(f"Mean edge similarity: {weights.mean():.3f}")


## 7. Detect communities recursively with Leiden

The first partition searches for broad communities. Each eligible child is then partitioned at a higher resolution, up to depth three. The resulting tree is data-driven: its branches need not reproduce the reference taxonomy exactly.


In [ ]:
tree = hierarchical_leiden(graph, leiden_config)

nodes_by_depth = Counter()
stack = [tree]
while stack:
    node = stack.pop()
    nodes_by_depth[node.depth] += 1
    stack.extend(node.children)

print(f"Nodes by depth: {dict(sorted(nodes_by_depth.items()))}")
assert max(nodes_by_depth) == 3, "The demonstration should reach three recursive levels."


## 8. Select representatives and inspect the tree

For every node, the pipeline averages its member embeddings, normalizes that centroid, and selects the closest original sentences. This is an extractive summary: representative text always comes directly from the corpus.


In [ ]:
add_representatives(tree, embeddings, n_representatives=3)
print_tree(tree, sentences, max_examples=1)


## 9. Compare clusters with the hidden reference taxonomy

The clustering is unsupervised, so evaluation happens only now. For each node, the table reports its dominant reference label and purity at the corresponding taxonomy level. Purity is descriptive here; the hierarchy may organize semantic themes differently from the synthetic labels.


In [ ]:
label_by_depth = {1: "habitat", 2: "animal_class", 3: "species"}

print(f"{'cluster':<18} {'depth':>5} {'size':>5} {'reference field':<16} {'dominant label':<12} {'purity':>7}")
print("-" * 72)
stack = list(reversed(tree.children))
while stack:
    node = stack.pop()
    field = label_by_depth.get(node.depth, "species")
    labels = [records[int(i)][field] for i in node.indices]
    dominant, count = Counter(labels).most_common(1)[0]
    print(f"{node.cluster_id:<18} {node.depth:>5} {node.size:>5} "
          f"{field:<16} {dominant:<12} {count / node.size:>7.1%}")
    stack.extend(reversed(node.children))


## 10. Prepare the hierarchy for export

`node_to_dict` resolves representative indices to text and creates a JSON-compatible nested object. Production code can persist this structure with `save_tree`; the notebook prints a compact preview to avoid creating generated files during every run.


In [ ]:
hierarchy = node_to_dict(tree, sentences)
print(json.dumps({
    "cluster_id": hierarchy["cluster_id"],
    "size": hierarchy["size"],
    "representative_sentences": hierarchy["representative_sentences"],
    "child_cluster_ids": [child["cluster_id"] for child in hierarchy["children"]],
}, indent=2))


## Next experiments

- Replace MiniLM with the default `all-mpnet-base-v2` model and compare partitions.
- Sweep `min_similarity` and inspect graph connectivity before clustering.
- Measure HNSW recall against exact neighbors on a sampled subset.
- Add domain metadata after clustering to characterize discovered communities.
- Export the tree with `src.cluster.save_tree` for visualization in another application.
